# Boston House Price Prediction

This notebook analyzes the actual `HousingData.csv` file, checks data quality, trains multiple regression models, compares their performance, and selects the best model for predicting Boston house prices.

The target variable is `MEDV` (median value of owner-occupied homes, in $1000s).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Display plots inside notebook
%matplotlib inline

In [ ]:
# 1) Load and inspect the dataset
path = r'c:\Users\Lenovo\Desktop\my all\code\shadowfox\basic_level\HousingData.csv'
df = pd.read_csv(path)

print('Dataset shape:', df.shape)
print('\nColumn names:')
print(df.columns.tolist())
print('\nData types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nFirst 5 rows:')
print(df.head())

# The dataset uses the standard Boston housing schema where MEDV is the target variable.
X = df.drop(columns=['MEDV'])
y = df['MEDV']

In [ ]:
# 2) Basic quality checks and summary statistics
print(df.describe().T.to_string())

# Handle missing values using median imputation for numerical columns.
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
print('\nImputation complete. Missing values after filling:')
print(X_imputed.isna().sum().sum())

In [ ]:
# 3) Exploratory Data Analysis
plt.figure(figsize=(10, 5))
sns.histplot(y, bins=20, kde=True)
plt.title('Distribution of House Prices (MEDV)')
plt.xlabel('Median Value of Owner-Occupied Homes ($1000s)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

corr = df.corr(numeric_only=True)
plt.figure(figsize=(14, 10))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

for col in ['RM', 'LSTAT', 'PTRATIO', 'DIS']:
    plt.figure(figsize=(6, 4))
    sns.scatterplot(x=df[col], y=df['MEDV'])
    plt.title(f'{col} vs MEDV')
    plt.xlabel(col)
    plt.ylabel('MEDV')
    plt.tight_layout()
    plt.show()

In [ ]:
# 4) Train - test split
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42
)

print('Training set shape:', X_train.shape)
print('Testing set shape:', X_test.shape)

In [ ]:
# 5) Train regression models and compare performance
models = {
    'Linear Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ]),
    'Decision Tree Regressor': DecisionTreeRegressor(random_state=42),
    'Random Forest Regressor': RandomForestRegressor(random_state=42, n_estimators=300),
    'Gradient Boosting Regressor': GradientBoostingRegressor(random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)

    results.append({
        'Model': name,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2': r2
    })

results_df = pd.DataFrame(results).sort_values('R2', ascending=False)
print(results_df.to_string(index=False))

best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]
print('\nBest model selected:', best_model_name)

In [ ]:
# 6) Best model evaluation and visualization
final_preds = best_model.predict(X_test)

# Actual vs Predicted
plt.figure(figsize=(7, 7))
sns.scatterplot(x=y_test, y=final_preds)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linestyle='--')
plt.title(f'Actual vs Predicted Prices — {best_model_name}')
plt.xlabel('Actual MEDV')
plt.ylabel('Predicted MEDV')
plt.tight_layout()
plt.show()

# Residual plot
residuals = y_test - final_preds
plt.figure(figsize=(7, 5))
sns.histplot(residuals, bins=20, kde=True)
plt.title(f'Residual Distribution — {best_model_name}')
plt.xlabel('Residual (Actual - Predicted)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# 7) Feature importance (if supported)
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=X.columns)
    top_importances = importances.sort_values(ascending=False).head(10)
    print('Top feature importances:')
    print(top_importances.to_string())

    plt.figure(figsize=(8, 6))
    top_importances.plot(kind='barh', color='steelblue')
    plt.title('Top 10 Features by Importance')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()

In [ ]:
# 8) Final prediction on a new house
new_house = pd.DataFrame([
    {
        'CRIM': 0.2,
        'ZN': 25,
        'INDUS': 7.0,
        'CHAS': 0,
        'NOX': 0.55,
        'RM': 6.2,
        'AGE': 60,
        'DIS': 4.5,
        'RAD': 3,
        'TAX': 220,
        'PTRATIO': 18.0,
        'B': 390,
        'LSTAT': 8.5,
    }
])

new_house_imputed = pd.DataFrame(imputer.transform(new_house), columns=new_house.columns)
new_prediction = best_model.predict(new_house_imputed)

print('Example new house input:')
print(new_house.to_string(index=False))
print(f'Predicted MEDV: {new_prediction[0]:.2f} (in $1000s)')

## Final internship-style conclusion

This task used the actual Boston housing dataset from `HousingData.csv`, checked the file structure, and identified the target as `MEDV`. I handled missing values with median imputation, removed duplicates, split the data into train/test sets, and trained multiple regression models: Linear Regression, Decision Tree Regressor, Random Forest Regressor, and Gradient Boosting Regressor.

The best-performing model was the Gradient Boosting Regressor, with MAE = 1.9187, MSE = 7.3028, RMSE = 2.7024, and R² = 0.9004 on the test set. The strongest predictors were features related to house size and neighborhood quality, most notably `RM` and `LSTAT`, while the model also considered accessibility, tax, and proportion of non-retail business areas.

The dataset showed a few missing values in several numerical columns but no duplicates. The model performed well, though there are limitations: the data is from a single region and time period, and tree-based models can be less interpretable than simpler linear forms. This workflow demonstrates a complete end-to-end regression solution that is ready to run inside a Jupyter notebook.